# Comprensione del dataset
Notebook di comprensione del dataset Paderborn: ogni sezione risponde a una domanda. La logica sta nei moduli in `src/`; qui orchestriamo e documentiamo.

## Il dataset
- **Chi lo ha creato:** Chair of Design and Drive Technology, Paderborn University (Lessmeier et al., 2016), KAT Bearing Data Center.
- **A cosa serve:** benchmark per la diagnosi di guasti ai cuscinetti di motori a induzione.
- **Come e' organizzato:** un archivio `.rar` per cuscinetto; dentro, le registrazioni `.mat` e i PDF descrittivi.
- **Quali classi contiene:** cuscinetti sani (`K`), guasto anello esterno (`KA`), interno (`KI`), combinato (`KB`).
- **Come sono nominati i file:** `Nxx_Mxx_Fxx_CODICE_i.mat`, cioe' `regime_cuscinetto_ripetizione`.

## Setup

In [ ]:
!apt-get -qq install -y unrar
!pip -q install scipy pandas pymupdf matplotlib

In [ ]:
import sys
from pathlib import Path

# rende importabili i moduli del progetto in Colab
if not Path('utils').exists():
    !git clone -q https://github.com/matpaol/bearings_detection.git
    %cd bearings_detection

from utils import config, repository, dataset, plots

## Come e' organizzato il repository remoto?
Non ci interessa il server in se', ma la struttura del dataset: quali cuscinetti esistono.

In [ ]:
codes = repository.list_available_bearings(config.BASE_URL)
print(len(codes), 'cuscinetti')
dataset.census(codes)

## Anatomia di un bearing
Cosa contiene un archivio, quanti file, e perche'. Scarichiamo un cuscinetto sano di riferimento.

In [ ]:
archive = repository.download_bearing('K001', config.DATA_DIR, config.BASE_URL)
folder = repository.extract_bearing(archive, config.DATA_DIR)

files = sorted(p.name for p in folder.rglob('*') if p.is_file())
from collections import Counter
suffixes = Counter(Path(f).suffix for f in files)
print('file totali:', len(files))
print('per estensione:', dict(suffixes))

Ci sono **80 `.mat`** = 4 regimi di carico x 20 ripetizioni, e **2 PDF**: il *profilo di danno* del cuscinetto (geometria) e il *measuring log* (setup e strumentazione).

## Come sono strutturati internamente i segnali?
Un `.mat` contiene una struct con i campi `Info / X / Y / Description`. In `Y` stanno i 7 segnali, ciascuno con nome, dati e metadati:

```
MAT -> struct -> Y (7 segnali) -> metadati -> sampling frequency
```

In [ ]:
recording = sorted(folder.rglob('N15_M07_F10_K001_*.mat'))[0]
dataset.signal_summary(recording)

## Qual e' l'unita' fondamentale del dataset?
La gerarchia reale, che guidera' il builder del notebook 02:

```
Bearing (K001)
  └─ Condition / regime (N09_M07_F10)      4 regimi
       └─ Measurement (_1 ... _20)          20 ripetizioni = un file .mat
            └─ Signals (7 canali)            corrente, vibrazione, speed, ...
```

L'unita' su disco e' quindi la **measurement** (un `.mat`), che contiene tutti i canali.

## Che aspetto hanno i segnali?
Visualizziamo corrente, vibrazione e velocita' della stessa registrazione. Useremo poi solo la corrente, ma qui documentiamo tutto.

In [ ]:
import matplotlib.pyplot as plt

current = dataset.load_signal(recording, 'phase_current_1')
vibration = dataset.load_signal(recording, 'vibration_1')
speed = dataset.load_signal(recording, 'speed')

fig, axes = plt.subplots(3, 1, figsize=(10, 7), constrained_layout=True)
plots.plot_signal(current, 64000, 'corrente di statore', seconds=0.05, ax=axes[0])
plots.plot_signal(vibration, 64000, 'vibrazione', seconds=0.05, ax=axes[1])
plots.plot_signal(speed, 4000, 'velocita', ax=axes[2])
plt.show()

## Come sono distribuite classi e natura?
La natura del danno (reale vs artificiale) non e' nel nome del file: viene dalle liste documentate.

In [ ]:
dataset.census(codes)

## Qual e' il contesto fisico?
I PDF danno la geometria del cuscinetto (per le frequenze caratteristiche BPFO/BPFI) e le varianti di carico. Materiale per la relazione, non per la pipeline.

In [ ]:
import fitz
for pdf_path in sorted(folder.rglob('*.pdf')):
    print(pdf_path.name)
    print(fitz.open(pdf_path)[0].get_text())